In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

In [2]:
months = ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6','Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']
features = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA', 'Percent_Bleaching']

In [5]:
df = pd.read_csv("../best_data.csv")

In [6]:
df = df[features + months]

In [7]:
X = df.drop("Percent_Bleaching", axis=1)
Y = df["Percent_Bleaching"]

In [8]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [9]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# OpTuna

In [17]:
# ── prepare inputs ──
month_cols     = ['Month_1','Month_2','Month_3','Month_4','Month_5','Month_6',
                  'Month_7','Month_8','Month_9','Month_10','Month_11','Month_12']
paper_features = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']

X_train_num    = X_train_raw[paper_features].copy()   # 5 numeric cols → impute + scale
X_extras_train = X_train_raw[month_cols].copy()       # 12 month OHE  → attach as-is

# ── CV helper: impute + scale numeric, then concat month OHE ──
def cv_with_month(model_factory, X_num, X_mon, y, cv=4):
    from sklearn.model_selection import KFold

    X_num = X_num.reset_index(drop=True)
    X_mon = X_mon.reset_index(drop=True)
    y     = y.reset_index(drop=True)

    kf     = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in kf.split(X_num):
        X_num_tr,  X_num_val = X_num.iloc[train_idx], X_num.iloc[val_idx]
        X_mon_tr  = X_mon.iloc[train_idx].reset_index(drop=True)
        X_mon_val = X_mon.iloc[val_idx].reset_index(drop=True)
        y_tr  = y.iloc[train_idx].reset_index(drop=True)
        y_val = y.iloc[val_idx].reset_index(drop=True)

        # ----- fit preprocessing on train fold only ⭐ -----
        imputer  = SimpleImputer(strategy='median')
        X_tr_imp = imputer.fit_transform(X_num_tr)
        X_vl_imp = imputer.transform(X_num_val)

        scaler  = StandardScaler()
        X_tr_ss = scaler.fit_transform(X_tr_imp)
        X_vl_ss = scaler.transform(X_vl_imp)

        # ----- concat scaled numeric + month OHE -----
        # [Lat, Lon, Depth, ClimSST, SSTA] + [Month_1 … Month_12] = 17 cols
        X_tr_final  = pd.concat(
            [pd.DataFrame(X_tr_ss, columns=paper_features), X_mon_tr], axis=1
        )
        X_vl_final  = pd.concat(
            [pd.DataFrame(X_vl_ss, columns=paper_features), X_mon_val], axis=1
        )

        # ----- train + evaluate -----
        model = model_factory()
        model.fit(X_tr_final, y_tr)
        scores.append(model.score(X_vl_final, y_val))   # R²

    return np.mean(scores)

In [18]:
def et_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 50, 300),
        max_depth        = trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5),
        max_features     = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1, random_state=42
    )
    return cv_with_month(
        lambda: ExtraTreesRegressor(**params),
        X_train_num,       # 5 numeric cols
        X_extras_train,    # 12 month OHE cols
        y_train
    )

et_study = optuna.create_study(direction="maximize")
et_study.optimize(et_objective, n_trials=50)
print("========== ET BEST ==========")
print(et_study.best_params)
print(f"Best R²: {et_study.best_value:.4f}")

[I 2026-05-17 11:12:11,929] A new study created in memory with name: no-name-aab0fade-90d0-4e14-9adc-97286c8508be
[I 2026-05-17 11:12:12,360] Trial 0 finished with value: 0.3961820598918606 and parameters: {'n_estimators': 76, 'max_depth': 34, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.3961820598918606.
[I 2026-05-17 11:12:13,034] Trial 1 finished with value: 0.2592382359725986 and parameters: {'n_estimators': 190, 'max_depth': 11, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.3961820598918606.
[I 2026-05-17 11:12:13,923] Trial 2 finished with value: 0.43193087720754475 and parameters: {'n_estimators': 203, 'max_depth': 25, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 2 with value: 0.43193087720754475.
[I 2026-05-17 11:12:15,855] Trial 3 finished with value: 0.6222535595421168 and parameters: {'n_estimators': 289, 'max_depth': 38, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 3 with valu

========== ET BEST ==========
{'n_estimators': 215, 'max_depth': 29, 'min_samples_leaf': 1, 'max_features': 'log2'}
Best R²: 0.6237


In [19]:
def rf_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 50, 300),
        max_depth         = trial.suggest_int("max_depth", 5, 40),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 5),
        max_features      = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1, random_state=42
    )
    return cv_with_month(
        lambda: RandomForestRegressor(**params),
        X_train_num,
        X_extras_train,
        y_train
    )

rf_study = optuna.create_study(direction="maximize")
rf_study.optimize(rf_objective, n_trials=50)
print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")


[I 2026-05-17 11:13:12,649] A new study created in memory with name: no-name-c9508721-d050-4034-8ee6-b8db42aabea8
[I 2026-05-17 11:13:14,060] Trial 0 finished with value: 0.4092200192224648 and parameters: {'n_estimators': 231, 'max_depth': 9, 'min_samples_split': 9, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.4092200192224648.
[I 2026-05-17 11:13:15,034] Trial 1 finished with value: 0.5890835934838174 and parameters: {'n_estimators': 111, 'max_depth': 26, 'min_samples_split': 9, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.5890835934838174.
[I 2026-05-17 11:13:16,293] Trial 2 finished with value: 0.557973917781126 and parameters: {'n_estimators': 145, 'max_depth': 17, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 1 with value: 0.5890835934838174.
[I 2026-05-17 11:13:17,497] Trial 3 finished with value: 0.5657306659046741 and parameters: {'n_estimators': 143, 'max_depth': 19, 'min

========== RF BEST ==========
{'n_estimators': 128, 'max_depth': 21, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
Best R²: 0.6241


In [20]:
def hgb_objective(trial):
    params = dict(
        max_iter          = trial.suggest_int  ("max_iter",          50, 400),
        max_depth         = trial.suggest_int  ("max_depth",          3,  15),
        learning_rate     = trial.suggest_float("learning_rate",  0.005, 0.1),
        min_samples_leaf  = trial.suggest_int  ("min_samples_leaf",   1,  50),
        l2_regularization = trial.suggest_float("l2_regularization", 0.0, 1.0),
        random_state=42
    )
    return cv_with_month(
        lambda: HistGradientBoostingRegressor(**params),
        X_train_num,
        X_extras_train,
        y_train
    )

hgb_study = optuna.create_study(direction="maximize")
hgb_study.optimize(hgb_objective, n_trials=50)
print("========== HGB BEST ==========")
print(hgb_study.best_params)
print(f"Best R²: {hgb_study.best_value:.4f}")

[I 2026-05-17 11:14:38,257] A new study created in memory with name: no-name-9db4c859-d940-4067-9f71-cf3f199ce9ed
[I 2026-05-17 11:14:40,055] Trial 0 finished with value: 0.5445666584131584 and parameters: {'max_iter': 251, 'max_depth': 12, 'learning_rate': 0.06947275092064441, 'min_samples_leaf': 7, 'l2_regularization': 0.2681687392962232}. Best is trial 0 with value: 0.5445666584131584.
[I 2026-05-17 11:14:41,638] Trial 1 finished with value: 0.4743837767788801 and parameters: {'max_iter': 219, 'max_depth': 12, 'learning_rate': 0.03241009455348857, 'min_samples_leaf': 36, 'l2_regularization': 0.44075810667381554}. Best is trial 0 with value: 0.5445666584131584.
[I 2026-05-17 11:14:42,736] Trial 2 finished with value: 0.43947078023097774 and parameters: {'max_iter': 145, 'max_depth': 13, 'learning_rate': 0.028329460016271485, 'min_samples_leaf': 38, 'l2_regularization': 0.44628751155282687}. Best is trial 0 with value: 0.5445666584131584.
[I 2026-05-17 11:14:44,269] Trial 3 finished w

========== HGB BEST ==========
{'max_iter': 355, 'max_depth': 13, 'learning_rate': 0.08264529806189247, 'min_samples_leaf': 7, 'l2_regularization': 0.25364870686226604}
Best R²: 0.5684


# Start

In [ ]:
# ========== ET BEST ==========
# {'n_estimators': 122, 'max_depth': 29, 'min_samples_leaf': 1, 'max_features': 'log2'}

# ========== RF BEST ==========
# {'n_estimators': 204, 'max_depth': 29, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}

# ========== HGB BEST ==========
# {'max_iter': 358, 'max_depth': 13, 'learning_rate': 0.07091369752390678, 'min_samples_leaf': 1, 'l2_regularization': 0.008609248841581194}


In [21]:
hgb_model = HistGradientBoostingRegressor(
    **hgb_study.best_params,
    random_state=42
)

et_model = ExtraTreesRegressor(
    **et_study.best_params,
    n_jobs=-1,
    random_state=42
)

rf_model = RandomForestRegressor(
    **rf_study.best_params,
    n_jobs=-1, 
    random_state=42
)

# K-Fold

In [22]:
from sklearn.model_selection import KFold
from sklearn.base import clone

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

month_cols     = ['Month_1','Month_2','Month_3','Month_4','Month_5','Month_6',
                  'Month_7','Month_8','Month_9','Month_10','Month_11','Month_12']
paper_features = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']

X_month = X[month_cols].copy()    # 12 month OHE  → attach as-is
X_num   = X[paper_features].copy() # 5 numeric     → impute + scale

cv_models = {
    'ExtraTrees':        et_model,
    'RandomForest':      rf_model,
    'HistGradientBoost': hgb_model,
    'Voting (Ensemble)': VotingRegressor([
        ('et',  et_model),
        ('rf',  rf_model),
        ('hgb', hgb_model)
    ])
}

print("=" * 70)
print("K-Fold Cross Validation Summary (k=5)")
print("=" * 70)
print(f"{'Model':<22}{'Mean R²':>14}{'Mean RMSE':>14}{'Mean MAE':>14}")
print("-" * 70)

for name, base_model in cv_models.items():
    r2_scores, rmse_scores, mae_scores = [], [], []

    for train_idx, test_idx in kf.split(X_num):
        # ----- split -----
        X_num_tr,  X_num_te  = X_num.iloc[train_idx],   X_num.iloc[test_idx]
        X_mon_tr             = X_month.iloc[train_idx].reset_index(drop=True)
        X_mon_te             = X_month.iloc[test_idx].reset_index(drop=True)
        y_fold_train         = Y.iloc[train_idx].reset_index(drop=True)
        y_fold_test          = Y.iloc[test_idx].reset_index(drop=True)

        # ----- fit preprocessing on train fold only ⭐ -----
        imputer  = SimpleImputer(strategy='median')
        X_tr_imp = imputer.fit_transform(X_num_tr)
        X_te_imp = imputer.transform(X_num_te)

        scaler  = StandardScaler()
        X_tr_ss = scaler.fit_transform(X_tr_imp)
        X_te_ss = scaler.transform(X_te_imp)

        # ----- concat scaled numeric + month OHE -----
        # [Lat, Lon, Depth, ClimSST, SSTA] + [Month_1 … Month_12] = 17 cols
        X_train_final = pd.concat(
            [pd.DataFrame(X_tr_ss, columns=paper_features), X_mon_tr], axis=1
        )
        X_test_final  = pd.concat(
            [pd.DataFrame(X_te_ss, columns=paper_features), X_mon_te], axis=1
        )

        # ----- train & evaluate -----
        model = clone(base_model)
        model.fit(X_train_final, y_fold_train)
        preds = model.predict(X_test_final)

        r2_scores.append(r2_score(y_fold_test, preds))
        rmse_scores.append(np.sqrt(mean_squared_error(y_fold_test, preds)))
        mae_scores.append(mean_absolute_error(y_fold_test, preds))

    print(f"{name:<22}{np.mean(r2_scores):>14.4f}{np.mean(rmse_scores):>14.4f}{np.mean(mae_scores):>14.4f}")

print("=" * 70)

K-Fold Cross Validation Summary (k=5)
Model                        Mean R²     Mean RMSE      Mean MAE
----------------------------------------------------------------------
ExtraTrees                    0.6694       12.3835        5.3824
RandomForest                  0.6610       12.5386        6.3401
HistGradientBoost             0.5844       13.8790        7.8381
Voting (Ensemble)             0.6647       12.4694        6.3896
